In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed
from datetime import datetime
from stockfish import Stockfish
import numpy as np
import os
import pandas as pd
import ast

In [ ]:
Annotated_cleaned= ** Data **

## Value

In [ ]:
STOCKFISH_PATH="D:\\MissTiny\\GitHub\\Creativity_Chess\\stockfish\\stockfish-windows-x86-64-avx2.exe"
EVAL_DEPTH=15
MAX_WORKERS=32

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from stockfish import Stockfish
from datetime import datetime
import os, re, math, numpy as np, pandas as pd

UCI_RE = re.compile(r"^[a-h][1-8][a-h][1-8][qrbn]?$")
def looks_uci(m): return bool(UCI_RE.fullmatch(str(m).strip()))

def ensure_uci(moves):
    if moves is None or (isinstance(moves, float) and math.isnan(moves)): return []
    if isinstance(moves, str): seq = moves.split()
    elif isinstance(moves, (list, tuple, np.ndarray, pd.Series)): seq = [str(x) for x in moves]
    else: seq = [str(x) for x in list(moves)]
    if seq and not all(looks_uci(x) for x in seq):
        import chess
        b = chess.Board(); out = []
        for s in seq:
            s = s.strip()
            m = b.parse_san(s) if not looks_uci(s) else chess.Move.from_uci(s)
            out.append(m.uci()); b.push(m)
        return out
    return seq

def cp(ev):  # map mate to big centipawns
    t, v = ev.get("type"), ev.get("value")
    if t == "cp":
        value = int(v)
    elif int(v)>0:
        value = int(100000*(1-int(v)/(EVAL_DEPTH+1)))
    else:
        value = int(-100000*(1-abs(int(v))/(EVAL_DEPTH+1)))
    
    return   value

def eval_row_thread(i, moves):
    sf = Stockfish(STOCKFISH_PATH, depth=EVAL_DEPTH)
    seq = ensure_uci(moves)
    evals, rels, prev = [], [], 0
    for j in range(len(seq)):
        sf.set_position(seq[:j+1])          # most stable across wrappers
        c = cp(sf.get_evaluation())
        rel = (c - prev) if (j % 2 == 0) else (-c + prev)
        prev = c
        evals.append(c); rels.append(rel)
    return i, evals, rels

def parallel_eval_threads(df, moves_col="Moves", max_workers=max(1, (os.cpu_count() or 2)-1)):
    n = len(df)
    values_cp, rel_vals = [None]*n, [None]*n
    print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Start")
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        fut2i = {pool.submit(eval_row_thread, i, df.iloc[i][moves_col]): i for i in range(n)}
        done = 0
        for fut in as_completed(fut2i):
            i = fut2i[fut]
            try:
                idx, evs, rels = fut.result()
                values_cp[idx] = evs; rel_vals[idx] = rels
            except Exception as e:
                values_cp[i] = []; rel_vals[i] = []
                print(f"[warn] row {i} failed: {e}")
            done += 1
            if done % 1000 == 0:
                print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: finished {done}/{n}")
    print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Ends")
    return values_cp, rel_vals

In [ ]:
vals, rels = parallel_eval_threads(Annotated_cleaned, moves_col="Moves", max_workers=MAX_WORKERS)

In [ ]:
Annotated_cleaned["values_15_cp"] = vals
Annotated_cleaned["relative_value_15_cp"] = rels